# 13 — Ekspor data untuk dashboard (Final Project)

Dashboard tidak membaca HDFS langsung. Notebook ini mengubah hasil Assignment II menjadi beberapa file ringkas di `docs/data/` (*serving layer*), dan memvalidasi setiap file terhadap angka Assignment II.

| File | Isi | Sumber |
|---|---|---|
| `predictions.csv` | satu baris per rekaman test | `/datalake/ml/predictions`, metadata, G1 |
| `waveforms_sample.json` | 48 sinyal asli untuk antrean pemeriksaan | `/datalake/ml/waveforms_z150s` |
| `stations.csv` | satu baris per stasiun | G1, G2, PageRank, label propagation |
| `explosions.csv` | lokasi kejadian ledakan | metadata |
| `summary.json` | angka ringkas, kelompok, gunung api, provenance | semua di atas |
| `states.json` | batas negara bagian WA/OR/ID | `us-atlas@3.0.1` |

Hasil lama tidak disentuh. Tulisan baru hanya ke `/datalake/dashboard/`, `docs/data/`, dan `outputs/dashboard/`.

In [1]:
import sys, os, json, subprocess, datetime, urllib.request
sys.path.insert(0, os.path.abspath('../src'))
from common import *
import ml_prep as mp
import numpy as np, pandas as pd
from pyspark.sql import functions as F

pd.set_option('display.width', 200, 'display.max_columns', 30)
spark = spark_session('a3_dashboard_export', packages='io.graphframes:graphframes-spark3_2.12:0.12.3')
spark.sparkContext.setCheckpointDir(os.path.expanduser('~/pnwml-data/tmp/graph-ckpt'))
from graphframes import GraphFrame

DASH = '/datalake/dashboard'
DATA, AUDIT = os.path.join(ROOT, 'docs', 'data'), os.path.join(ROOT, 'outputs', 'dashboard')
os.makedirs(DATA, exist_ok=True); os.makedirs(AUDIT, exist_ok=True)
TZ = 'America/Los_Angeles'
EVENTS = ['earthquake', 'explosion', 'surface_event']
PCOLS = [f'p_{c}' for c in CLASSES]
R = lambda p: spark.read.parquet(f'{HDFS}{p}')

CHECKS, PROV = [], {}
def check(name, ok, detail='', hard=True):
    """Catat hasil pemeriksaan. Hard FAIL menghentikan notebook."""
    ok = bool(ok)
    CHECKS.append((name, 'PASS' if ok else 'FAIL', str(detail)))
    print(f"{'PASS' if ok else 'FAIL'}  {name}" + (f'  [{detail}]' if detail != '' else ''))
    if hard and not ok:
        raise RuntimeError(f'hard FAIL: {name}: {detail}')

def read_md(name):
    """Baca tabel markdown di outputs/tables menjadi DataFrame (semua kolom teks)."""
    rows = [[c.strip() for c in l.strip().strip('|').split('|')] for l in open(f'{OUT_TABLES}/{name}.md') if l.startswith('|')]
    return pd.DataFrame(rows[2:], columns=rows[0])

meta, e1, v2, e2 = R(f'{PROCESSED}/metadata'), R('/datalake/graph/g1_edges'), R('/datalake/graph/g2_vertices'), R('/datalake/graph/g2_edges')

:: loading settings :: url = jar:file:/Users/axelnakata/Documents/BigData-Earthquake/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/axelnakata/.ivy2/cache
The jars for the packages stored in: /Users/axelnakata/.ivy2/jars
io.graphframes#graphframes-spark3_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4829fa4c-ab61-4048-ae9d-10d9249bce0d;1.0
	confs: [default]
	found io.graphframes#graphframes-spark3_2.12;0.12.3 in central
	found io.graphframes#graphframes-graphx-spark3_2.12;0.12.3 in central
	found org.apache.datasketches#datasketches-java;6.2.0 in central
	found org.apache.datasketches#datasketches-memory;3.0.2 in central
:: resolution report :: resolve 74ms :: artifacts dl 2ms
	:: modules in use:
	io.graphframes#graphframes-graphx-spark3_2.12;0.12.3 from central in [default]
	io.graphframes#graphframes-spark3_2.12;0.12.3 from central in [default]
	org.apache.datasketches#datasketches-java;6.2.0 from central in [default]
	org.apache.datasketches#datasketches-memory;3.0.2 from central in [default]
	----------------------------------------

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 12:20:10 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


Folder baru di HDFS memakai pola hak akses yang sama dengan `/datalake/graph` (notebook 12 sel 3).

In [2]:
sh(f'hdfs dfs -mkdir -p {DASH} && hdfs dfs -chgrp engineer {DASH} && hdfs dfs -chmod 770 {DASH}')
sh(f'hdfs dfs -setfacl -m group:scientist:r-x,default:group:engineer:rwx,default:group:scientist:r-x,default:other::--- {DASH}');

$ hdfs dfs -mkdir -p /datalake/dashboard && hdfs dfs -chgrp engineer /datalake/dashboard && hdfs dfs -chmod 770 /datalake/dashboard



$ hdfs dfs -setfacl -m group:scientist:r-x,default:group:engineer:rwx,default:group:scientist:r-x,default:other::--- /datalake/dashboard



## 1. `predictions.csv`
Prediksi data test digabung dengan metadata (sensor, stasiun, waktu lokal) dan derajat kejadian di G1 (`n_stations_event`). Waktu UTC diubah ke waktu Pasifik dengan `from_utc_timestamp`, jadi DST ikut terhitung.

In [3]:
degree = e1.groupBy('src').agg(F.count('*').alias('n_stations_event'))
p = (R(f'{ML}/predictions').filter("split = 'test'").drop('split')
     .join(meta.select('dataset', 'trace_name', 'event_id', 'station_network_code', 'station_code', 'station_channel_code', 'trace_start_time'),
           ['dataset', 'trace_name'], 'left')
     .withColumn('src', F.when(F.col('event_id').isNotNull(), F.concat_ws(':', F.lit('ev'), 'dataset', 'event_id')))
     .join(degree, 'src', 'left')
     .select('dataset', 'trace_name', 'label', 'pred', *PCOLS,
             F.col('station_channel_code').alias('sensor'),
             F.concat_ws('.', 'station_network_code', 'station_code').alias('station'),
             F.date_format(F.from_utc_timestamp('trace_start_time', TZ), 'yyyy-MM-dd HH:mm').alias('local_time'),
             'n_stations_event')
     .toPandas().sort_values(['dataset', 'trace_name'], kind='mergesort').reset_index(drop=True))
p['rid'] = p.index
p['n_stations_event'] = p.n_stations_event.astype('Int64')

n_test = R(f'{ML}/predictions').filter("split = 'test'").count()
check('predictions: jumlah baris = baris test HDFS', len(p) == n_test == 35341, f'{len(p):,}')
raw = p[PCOLS].to_numpy(np.float64)
check('predictions: pred = argmax', (np.array(CLASSES)[raw.argmax(1)] == p.pred.to_numpy()).all())
check('predictions: jumlah peluang = 1 (±0,01)', (np.abs(raw.sum(1) - 1) <= 0.01).all(), f'{raw.sum(1).min():.6f}..{raw.sum(1).max():.6f}')

cm = pd.crosstab(p.label, p.pred).reindex(index=CLASSES, columns=CLASSES).fillna(0).astype(int)
ref = read_md('a2_confusion_matrix').set_index('asli \\ tebakan').loc[CLASSES, CLASSES].astype(int)
check('predictions: confusion matrix = a2_confusion_matrix.md', (cm.to_numpy() == ref.to_numpy()).all())
m = cm.to_numpy(); tp = np.diag(m); pr, rc = tp / m.sum(0), tp / m.sum(1); f1 = 2 * pr * rc / (pr + rc)
acc, macro = tp.sum() / m.sum(), f1.mean()
check('predictions: accuracy 0,925 (±0,001)', abs(acc - 0.925) <= 0.001, f'{acc:.4f}')
check('predictions: macro-F1 0,861 (±0,001)', abs(macro - 0.861) <= 0.001, f'{macro:.4f}')
isnoise = p.label == 'noise'
check('predictions: n_stations_event kosong hanya untuk noise', (p.n_stations_event.isna() == isnoise).all())
check('predictions: n_stations_event ≥ 1 untuk kejadian', (p.loc[~isnoise, 'n_stations_event'] >= 1).all())
check('predictions: sensor hanya EH/BH/HH', set(p.sensor) <= {'EH', 'BH', 'HH'})

for c in PCOLS:
    p[c] = p[c].astype(float).round(3)
p[['rid', 'dataset', 'trace_name']].to_csv(f'{AUDIT}/rid_map.csv', index=False)
cols = ['rid', 'label', 'pred', *PCOLS, 'sensor', 'station', 'local_time', 'n_stations_event']
p[cols].to_csv(f'{DATA}/predictions.csv', index=False, float_format='%.3f')
PROV['predictions.csv'] = {'source': [f'{ML}/predictions (split=test)', f'{PROCESSED}/metadata', '/datalake/graph/g1_edges'], 'rows': len(p)}
cm

PASS  predictions: jumlah baris = baris test HDFS  [35,341]
PASS  predictions: pred = argmax
PASS  predictions: jumlah peluang = 1 (±0,01)  [1.000000..1.000000]
PASS  predictions: confusion matrix = a2_confusion_matrix.md
PASS  predictions: accuracy 0,925 (±0,001)  [0.9249]
PASS  predictions: macro-F1 0,861 (±0,001)  [0.8609]
PASS  predictions: n_stations_event kosong hanya untuk noise
PASS  predictions: n_stations_event ≥ 1 untuk kejadian
PASS  predictions: sensor hanya EH/BH/HH


pred,earthquake,explosion,surface_event,noise
label,,,,
earthquake,22296,944,266,431
explosion,188,2174,28,17
surface_event,150,79,1137,15
noise,390,116,31,7079


## 2. `waveforms_sample.json`
48 rekaman test: untuk tiap kelas analis, 3 rekaman dari tiap keranjang keyakinan. Sinyal Z dipotong ke detik 15–135 (potongan evaluasi model), lalu diringkas 12.000 → 600 titik. Tiap kelompok 20 sampel menyimpan nilai dengan amplitudo terbesar, supaya lonjakan tidak hilang.

In [4]:
p['conf'] = p[PCOLS].max(axis=1)
right = p.label == p.pred
BUCKETS = {'a_benar_conf>=0.95': right & (p.conf >= .95), 'b_benar_conf_0.70-0.95': right & p.conf.between(.70, .95, inclusive='left'),
           'c_conf<0.70': p.conf < .70, 'd_salah_conf>=0.85': ~right & (p.conf >= .85)}
picked, cover = [], []
for c in CLASSES:
    for b, mask in BUCKETS.items():
        pool = p[(p.label == c) & mask].sort_values('rid')
        take = pool.sample(min(3, len(pool)), random_state=42)
        picked.append(take); cover.append({'kelas': c, 'keranjang': b, 'tersedia': len(pool), 'diambil': len(take)})
picked = pd.concat(picked); cover = pd.DataFrame(cover)
print('kurang dari 3 di:', cover[cover.diambil < 3].to_dict('records'))

def to_points(z):
    """15.000 sampel → potongan evaluasi 12.000 → 600 titik (peak-preserving)."""
    b = mp.crop(np.asarray(z, dtype=np.float64), False).reshape(600, 20)
    return np.round(b[np.arange(600), np.abs(b).argmax(1)], 2).tolist()

assert mp.N == 15000 and mp.CROP == 12000 and (mp.N - mp.CROP) // 2 == 1500
keys = spark.createDataFrame(picked[['dataset', 'trace_name', 'rid']].astype({'rid': int}))
wz = (R(f'{ML}/waveforms_z150s').filter("split = 'test'").select('dataset', 'trace_name', 'z')
      .join(F.broadcast(keys), ['dataset', 'trace_name']).select('rid', 'z').toPandas())
sample = [{'rid': int(r.rid), 'points': to_points(r.z)} for r in wz.sort_values('rid').itertuples()]
check('waveforms: jumlah sesuai keranjang', len(sample) == int(cover.diambil.sum()), f'{len(sample)} dari 48')
check('waveforms: tiap entri 600 titik', all(len(s['points']) == 600 for s in sample))
check('waveforms: semua rid ada di predictions', set(s['rid'] for s in sample) <= set(p.rid))
json.dump(sample, open(f'{DATA}/waveforms_sample.json', 'w'), separators=(',', ':'))
PROV['waveforms_sample.json'] = {'source': [f'{ML}/waveforms_z150s (split=test)'], 'rows': len(sample), 'buckets_short': cover[cover.diambil < 3].to_dict('records')}
cover.pivot(index='kelas', columns='keranjang', values='diambil').loc[CLASSES]

kurang dari 3 di: []


PASS  waveforms: jumlah sesuai keranjang  [48 dari 48]
PASS  waveforms: tiap entri 600 titik
PASS  waveforms: semua rid ada di predictions


keranjang,a_benar_conf>=0.95,b_benar_conf_0.70-0.95,c_conf<0.70,d_salah_conf>=0.85
kelas,,,,
earthquake,3,3,3,3
explosion,3,3,3,3
surface_event,3,3,3,3
noise,3,3,3,3


## 3. `stations.csv`
PageRank dan label propagation dijalankan ulang persis seperti notebook 12 (sel 16 dan 19). Hasil mentah label propagation disimpan di path baru. Hasilnya dibandingkan dengan pemetaan lama di `/datalake/graph/g2_communities`.

In [5]:
e2_both = e2.unionByName(e2.select(F.col('dst').alias('src'), F.col('src').alias('dst'), 'weight'))
g2 = GraphFrame(v2, e2_both)
pr_df = g2.pageRank(resetProbability=0.15, maxIter=20).vertices.select('id', 'pagerank').toPandas()
g2.labelPropagation(maxIter=10).select('id', 'label').write.mode('overwrite').parquet(f'{HDFS}{DASH}/lpa_rerun')
lp = R(f'{DASH}/lpa_rerun').toPandas()
nb = e2_both.groupBy('src').count().toPandas().rename(columns={'src': 'id', 'count': 'n_linked'})
rec = e1.groupBy('dst', 'label').agg(F.sum('recordings').alias('n')).toPandas().pivot(index='dst', columns='label', values='n').reindex(columns=EVENTS).fillna(0)
n_ev = e1.groupBy('dst').count().toPandas().rename(columns={'dst': 'id', 'count': 'n_events'})

st = (v2.toPandas().merge(lp, on='id').merge(pr_df, on='id').merge(nb, on='id', how='left').merge(n_ev, on='id', how='left')
      .fillna({'n_linked': 0, 'n_events': 0}))
st[['n_linked', 'n_events']] = st[['n_linked', 'n_events']].astype(int)
assert st.id.is_unique
linked = st.n_linked > 0
sizes = st[linked].label.value_counts()
names = {c: f'K{i + 1}' for i, c in enumerate(sizes.index[:8])}
st['group'] = np.where(~linked, 'terisolasi', st.label.map(names).fillna('lainnya'))
old = R('/datalake/graph/g2_communities').toPandas()
same = st[['id', 'label']].merge(old, on='id', suffixes=('', '_old'))
check('LPA: sama dengan pemetaan lama g2_communities', (same.label == same.label_old).all(), f'{(same.label == same.label_old).mean():.1%} stasiun', hard=False)

st = st.sort_values(['pagerank', 'name'], ascending=[False, True]).reset_index(drop=True)
st['pagerank_rank'] = st.index + 1
pct = (100 * rec.div(rec.sum(axis=1), axis=0)).round(1).add_prefix('pct_').reset_index().rename(columns={'dst': 'id'})
st = st.merge(pct, on='id', how='left')
top10 = read_md('a2_graph_top_pagerank')
check('stations: 642 baris', len(st) == 642, len(st))
check('stations: semua punya koordinat', st[['lat', 'lon']].notna().all().all())
check('PageRank: 10 teratas sama, urutan sama', st.name.head(10).tolist() == top10.name.tolist(), ', '.join(st.name.head(10)))
check('PageRank: skor ±0,01', (np.abs(st.pagerank.head(10).to_numpy() - top10.pagerank.astype(float).to_numpy()) <= 0.01).all())

out = st.rename(columns={'name': 'station'})[['station', 'network', 'lat', 'lon', 'group', 'n_events', 'n_linked', 'pagerank', 'pagerank_rank',
                                             'pct_earthquake', 'pct_explosion', 'pct_surface_event']].copy()
out[['lat', 'lon', 'pagerank']] = out[['lat', 'lon', 'pagerank']].round(3)
out.to_csv(f'{DATA}/stations.csv', index=False)
PROV['stations.csv'] = {'source': ['/datalake/graph/g1_edges', '/datalake/graph/g2_vertices', '/datalake/graph/g2_edges', f'{DASH}/lpa_rerun'], 'rows': len(out)}
out.head(3)

PASS  LPA: sama dengan pemetaan lama g2_communities  [100.0% stasiun]
PASS  stations: 642 baris  [642]
PASS  stations: semua punya koordinat
PASS  PageRank: 10 teratas sama, urutan sama  [UW.GNW, PB.B201, PB.B018, UW.HDW, PB.B941, UW.LTY, PB.B202, PB.B203, UW.DOSE, PB.B013]
PASS  PageRank: skor ±0,01


,station,network,lat,lon,group,n_events,n_linked,pagerank,pagerank_rank,pct_earthquake,pct_explosion,pct_surface_event
0,UW.GNW,UW,47.564,-122.825,K1,4653,161,4.433,1,96.0,4.0,0.0
1,PB.B201,PB,46.303,-122.265,K2,6658,154,4.288,2,92.2,6.9,0.8
2,PB.B018,PB,46.980,-123.020,K1,1968,160,4.232,3,90.7,9.3,0.0


Aturan toleransi yang disepakati pemilik (PRD §6.3): tetap 8 kelompok utama, ukuran dalam ±10% atau ±3 stasiun, lokasi rata-rata dekat, dan karakter kelompok tetap sama.

In [6]:
REGION = {'K1': 'Washington barat (Puget Sound)', 'K2': 'Washington barat daya – Oregon utara', 'K3': 'Washington timur', 'K4': 'Oregon barat daya',
          'K5': 'Oregon barat tengah', 'K6': 'Oregon tengah (kawasan gunung api)', 'K7': 'Oregon timur laut', 'K8': 'Perbatasan Oregon–Idaho',
          'lainnya': 'Lainnya', 'terisolasi': 'Terisolasi'}
KEYS = [f'K{i}' for i in range(1, 9)] + ['lainnya', 'terisolasi']
rec_g = rec.reset_index().rename(columns={'dst': 'id'}).merge(st[['id', 'group']], on='id').groupby('group')[EVENTS].sum()
grp = st.groupby('group').agg(stations=('id', 'size'), lat=('lat', 'mean'), lon=('lon', 'mean')).join(rec_g.sum(axis=1).rename('recordings')).join((100 * rec_g.div(rec_g.sum(axis=1), axis=0)).round(1))
ref = read_md('a2_graph_communities').set_index('komunitas')

check('LPA: tetap 8 kelompok utama + 2 kecil', len(sizes) == 10, f'{len(sizes)} kelompok terhubung')
for k in KEYS:
    want, got = int(ref.loc[k, 'stasiun']), int(grp.loc[k, 'stations'])
    check(f'LPA {k}: ukuran dalam toleransi', abs(got - want) <= max(.1 * want, 3), f'{got} vs {want}')
    if k != 'terisolasi':
        d = max(abs(grp.loc[k, 'lat'] - float(ref.loc[k, 'lat'])), abs(grp.loc[k, 'lon'] - float(ref.loc[k, 'lon'])))
        check(f'LPA {k}: lokasi rata-rata dekat', d <= .5, f'selisih {d:.2f}°')
check('LPA: K4 dan K5 kaya ledakan', grp.loc[['K4', 'K5'], 'explosion'].min() > 25, grp.loc[['K4', 'K5'], 'explosion'].tolist())
check('LPA: K6 punya surface event tertinggi', grp.loc[KEYS[:8], 'surface_event'].idxmax() == 'K6', grp.loc['K6', 'surface_event'])
grp.loc[KEYS].round(2)

PASS  LPA: tetap 8 kelompok utama + 2 kecil  [10 kelompok terhubung]
PASS  LPA K1: ukuran dalam toleransi  [246 vs 246]
PASS  LPA K1: lokasi rata-rata dekat  [selisih 0.00°]
PASS  LPA K2: ukuran dalam toleransi  [125 vs 125]
PASS  LPA K2: lokasi rata-rata dekat  [selisih 0.00°]
PASS  LPA K3: ukuran dalam toleransi  [61 vs 61]
PASS  LPA K3: lokasi rata-rata dekat  [selisih 0.00°]
PASS  LPA K4: ukuran dalam toleransi  [48 vs 48]
PASS  LPA K4: lokasi rata-rata dekat  [selisih 0.00°]
PASS  LPA K5: ukuran dalam toleransi  [24 vs 24]
PASS  LPA K5: lokasi rata-rata dekat  [selisih 0.00°]
PASS  LPA K6: ukuran dalam toleransi  [22 vs 22]
PASS  LPA K6: lokasi rata-rata dekat  [selisih 0.00°]
PASS  LPA K7: ukuran dalam toleransi  [12 vs 12]
PASS  LPA K7: lokasi rata-rata dekat  [selisih 0.00°]
PASS  LPA K8: ukuran dalam toleransi  [7 vs 7]
PASS  LPA K8: lokasi rata-rata dekat  [selisih 0.00°]
PASS  LPA lainnya: ukuran dalam toleransi  [10 vs 10]
PASS  LPA lainnya: lokasi rata-rata dekat  [selisih

,stations,lat,lon,recordings,earthquake,explosion,surface_event
group,,,,,,,
K1,246,47.74,-122.47,102537.0,90.2,6.9,2.9
K2,125,45.76,-122.58,62972.0,84.3,7.2,8.6
K3,61,46.39,-119.26,11304.0,86.8,12.9,0.4
K4,48,42.89,-123.07,4066.0,62.7,37.3,0.0
K5,24,43.97,-122.43,2962.0,53.5,44.2,2.4
K6,22,43.45,-120.86,2468.0,76.1,6.6,17.4
K7,12,45.21,-120.52,724.0,97.9,2.1,0.0
K8,7,44.48,-116.74,113.0,100.0,0.0,0.0
lainnya,10,44.73,-119.24,202.0,92.1,7.9,0.0


## 4. `explosions.csv`
Satu baris per kejadian ledakan. Lalu dilihat apakah lokasinya mengumpul di sekitar kelompok yang kaya ledakan (K4, K5).

In [7]:
ex = (meta.filter("label = 'explosion'").groupBy('event_id')
      .agg(F.first('source_latitude_deg').alias('lat'), F.first('source_longitude_deg').alias('lon'),
           F.first('preferred_source_magnitude').alias('magnitude'),
           F.date_format(F.from_utc_timestamp(F.first('source_origin_time'), TZ), 'yyyy-MM-dd HH:mm').alias('local_time'))
      .toPandas().sort_values('event_id').reset_index(drop=True))
g1_ex = e1.filter("label = 'explosion'").select('src').distinct().count()
check('explosions: jumlah ≈ 6.392 (kejadian ledakan di G1)', len(ex) == g1_ex == 6392, f'{len(ex):,}')
check('explosions: semua punya koordinat', ex[['lat', 'lon']].notna().all().all())
ex[['lat', 'lon']] = ex[['lat', 'lon']].round(3)
ex['magnitude'] = ex.magnitude.round(2)
ex.to_csv(f'{DATA}/explosions.csv', index=False)
PROV['explosions.csv'] = {'source': [f'{PROCESSED}/metadata (label=explosion)'], 'rows': len(ex)}

km = lambda lat1, lon1, lat2, lon2: 111.2 * np.hypot(np.subtract(lat1, lat2), np.subtract(lon1, lon2) * np.cos(np.radians(45.5)))
ex['cell'] = (ex.lat * 4).apply(np.floor) / 4 ; ex['cell_lon'] = (ex.lon * 4).apply(np.floor) / 4
cells = ex.groupby(['cell', 'cell_lon']).agg(n=('event_id', 'size'), lat=('lat', 'mean'), lon=('lon', 'mean')).sort_values('n', ascending=False)
cells['pct'] = (100 * cells.n / len(ex)).round(1)
print('sel 0,25° terpadat (dari', len(cells), 'sel):'); display(cells.head(15).round(2))

# Ledakan yang direkam stasiun tiap kelompok: seberapa dekat dengan pusat kelompok, dan seberapa terkumpul?
ev_st = (e1.filter("label = 'explosion'").select(F.split('src', ':').getItem(2).alias('event_id'), F.col('dst').alias('id')).toPandas()
         .merge(st[['id', 'group']], on='id').drop_duplicates(['event_id', 'group']).merge(ex, on='event_id'))
rows = []
for k in KEYS[:8]:
    g = ev_st[ev_st.group == k]
    if g.empty: continue
    c = g.groupby(['cell', 'cell_lon']).size().sort_values(ascending=False)
    d = km(g.lat, g.lon, grp.loc[k, 'lat'], grp.loc[k, 'lon'])
    rows.append({'kelompok': k, 'ledakan': len(g), 'median_km_ke_pusat': round(float(np.median(d)), 0), 'pct_dalam_100km': round(float((d <= 100).mean()) * 100, 1),
                 'lat_median': g.lat.median(), 'lon_median': g.lon.median(), 'sel_terpadat_pct': round(100 * c.iloc[0] / len(g), 1), 'pct_3_sel_terpadat': round(100 * c.iloc[:3].sum() / len(g), 1)})
ex_by_group = pd.DataFrame(rows).round(2)
ex_by_group

PASS  explosions: jumlah ≈ 6.392 (kejadian ledakan di G1)  [6,392]
PASS  explosions: semua punya koordinat
sel 0,25° terpadat (dari 459 sel):


n    lat     lon   pct
cell  cell_lon                          
49.25 -120.75   951  49.41 -120.53  14.9
      -120.50   381  49.39 -120.48   6.0
48.00 -122.00   293  48.10 -121.93   4.6
46.75 -122.25   170  46.95 -122.21   2.7
47.00 -122.00   157  47.16 -121.85   2.5
45.75 -122.50   137  45.84 -122.45   2.1
46.00 -123.25   116  46.19 -123.05   1.8
49.00 -122.25   116  49.11 -122.14   1.8
45.50 -122.50   110  45.68 -122.37   1.7
46.00 -123.00    90  46.11 -122.85   1.4
44.00 -123.25    83  44.11 -123.07   1.3
46.25 -122.50    81  46.37 -122.39   1.3
      -122.75    78  46.37 -122.60   1.2
48.25 -123.75    76  48.43 -123.54   1.2
47.25 -118.00    62  47.37 -117.88   1.0

,kelompok,ledakan,median_km_ke_pusat,pct_dalam_100km,lat_median,lon_median,sel_terpadat_pct,pct_3_sel_terpadat
0,K1,3326,171.0,31.7,48.49,-121.39,28.6,48.8
1,K2,1826,71.0,71.4,45.84,-122.63,7.5,19.8
2,K3,621,95.0,54.8,46.42,-119.18,9.8,24.3
3,K4,729,77.0,70.5,43.43,-123.11,8.1,23.9
4,K5,724,65.0,82.9,44.10,-122.89,10.6,22.2
5,K6,100,98.0,51.0,44.04,-121.02,18.0,42.0
6,K7,14,78.0,71.4,45.66,-121.09,14.3,28.6


In [8]:
VOLCANOES = [('Mount Baker', 48.777, -121.813), ('Glacier Peak', 48.112, -121.113), ('Mount Rainier', 46.853, -121.760), ('Mount Adams', 46.206, -121.490),
             ('Mount St. Helens', 46.20, -122.18), ('Mount Hood', 45.374, -121.695), ('Mount Jefferson', 44.674, -121.799), ('Three Sisters', 44.10, -121.77),
             ('Newberry Volcano', 43.722, -121.229), ('Crater Lake (Mount Mazama)', 42.93, -122.12), ('Mount McLoughlin', 42.445, -122.315)]
checks_v = []
for k, vs in {'K2': ['Mount St. Helens', 'Mount Hood'], 'K6': ['Newberry Volcano', 'Three Sisters']}.items():
    g = st[st.group == k]
    for n, la, lo in [v for v in VOLCANOES if v[0] in vs]:
        d = km(g.lat, g.lon, la, lo)
        checks_v.append({'kelompok': k, 'gunung': n, 'km_pusat_kelompok': round(float(km(grp.loc[k, 'lat'], grp.loc[k, 'lon'], la, lo)), 0),
                         'stasiun_≤30km': int((d <= 30).sum()), 'stasiun_≤50km': int((d <= 50).sum()), 'dari': len(g)})
volc_check = pd.DataFrame(checks_v)
volc_check

,kelompok,gunung,km_pusat_kelompok,stasiun_≤30km,stasiun_≤50km,dari
0,K2,Mount St. Helens,58.0,35,44,125
1,K2,Mount Hood,81.0,12,17,125
2,K6,Three Sisters,101.0,0,1,22
3,K6,Newberry Volcano,42.0,14,17,22


## 5. `summary.json` dan `states.json`
Definisi jam kerja mengikuti notebook 07 sel 20: hari Senin–Jumat dan `hour` 8–16 waktu Pasifik, jadi 08.00–16.59.

In [9]:
BUCKET_LABEL = ['1', '2-3', '4-9', '>=10']
deg = e1.groupBy('src', 'label').agg(F.count('*').alias('k')).toPandas()
deg['b'] = pd.cut(deg.k, [0, 1, 3, 9, 10**6], labels=BUCKET_LABEL)
per_event = {'buckets': BUCKET_LABEL}
PRD_SPE = {'earthquake': (39631, [.175, .328, .446, .051]), 'explosion': (6392, [.337, .444, .214, .005]), 'surface_event': (5425, [.777, .126, .091, .006])}
for c in EVENTS:
    d = deg[deg.label == c]
    share = (d.b.value_counts(normalize=True).reindex(BUCKET_LABEL).fillna(0).round(3)).tolist()
    per_event[c] = {'events': len(d), 'share': share}
    check(f'stations_per_event {c}: jumlah kejadian', len(d) == PRD_SPE[c][0], len(d))
    check(f'stations_per_event {c}: share ±0,002', max(abs(a - b) for a, b in zip(share, PRD_SPE[c][1])) <= .002, share)

t = F.from_utc_timestamp('trace_start_time', TZ)
office = {r.label: round(r.s, 3) for r in meta.groupBy('label').agg(F.avg((F.hour(t).between(8, 16) & F.dayofweek(t).between(2, 6)).cast('int')).alias('s')).collect()}
ref_o = read_md('a2_context_by_class').set_index('label').pct_weekday_08_17.astype(float) / 100
for c in CLASSES:
    check(f'office_hours {c}: sama dengan a2_context_by_class.md', abs(office[c] - ref_o[c]) <= .001, f'{office[c]} vs {ref_o[c]:.3f}')

total = rec_g.sum()
net = (100 * total / total.sum()).round(1)
check('network_average: rekaman = 187.751', int(total.sum()) == 187751, int(total.sum()))
print('rata-rata jaringan:', net.to_dict(), '(PRD: 86,7 / 8,6 / 4,8)')
for c, v in {'earthquake': 86.7, 'explosion': 8.6, 'surface_event': 4.8}.items():
    check(f'network_average {c} vs PRD ±0,05', abs(net[c] - v) <= .05, f'{net[c]} vs {v}', hard=False)

groups = [{'key': k, 'name_id': REGION[k], 'stations': int(grp.loc[k, 'stations']), 'lat': round(float(grp.loc[k, 'lat']), 2), 'lon': round(float(grp.loc[k, 'lon']), 2),
           'recordings': int(grp.loc[k, 'recordings']), 'pct': {c: float(grp.loc[k, c]) for c in EVENTS}} for k in KEYS]

PASS  stations_per_event earthquake: jumlah kejadian  [39631]
PASS  stations_per_event earthquake: share ±0,002  [[0.175, 0.328, 0.446, 0.051]]
PASS  stations_per_event explosion: jumlah kejadian  [6392]
PASS  stations_per_event explosion: share ±0,002  [[0.337, 0.444, 0.214, 0.005]]
PASS  stations_per_event surface_event: jumlah kejadian  [5425]
PASS  stations_per_event surface_event: share ±0,002  [[0.777, 0.126, 0.091, 0.007]]
PASS  office_hours earthquake: sama dengan a2_context_by_class.md  [0.249 vs 0.249]
PASS  office_hours explosion: sama dengan a2_context_by_class.md  [0.878 vs 0.878]
PASS  office_hours surface_event: sama dengan a2_context_by_class.md  [0.255 vs 0.255]
PASS  office_hours noise: sama dengan a2_context_by_class.md  [0.298 vs 0.298]
PASS  network_average: rekaman = 187.751  [187751]
rata-rata jaringan: {'earthquake': 86.7, 'explosion': 8.6, 'surface_event': 4.7} (PRD: 86,7 / 8,6 / 4,8)
PASS  network_average earthquake vs PRD ±0,05  [86.7 vs 86.7]
PASS  network_a

In [10]:
us = json.load(urllib.request.urlopen('https://cdn.jsdelivr.net/npm/us-atlas@3.0.1/states-10m.json'))
sx, sy = us['transform']['scale']; tx, ty = us['transform']['translate']
arcs = []
for a in us['arcs']:
    x = y = 0; pts = []
    for dx, dy in a:
        x += dx; y += dy; pts.append((x * sx + tx, y * sy + ty))
    arcs.append(pts)

def ring(ids):
    out = []
    for i in ids:
        pts = arcs[i] if i >= 0 else arcs[~i][::-1]
        out += pts[1:] if out else pts
    return out

def rounded(r, nd):
    """Bulatkan koordinat, buang titik berurutan yang jadi sama."""
    out = []
    for x, y in r:
        q = [round(x, nd), round(y, nd)]
        if not out or q != out[-1]:
            out.append(q)
    return out

wanted = {'53': 'Washington', '41': 'Oregon', '16': 'Idaho'}
geoms = [g for g in us['objects']['states']['geometries'] if g['id'] in wanted]
check('states: 3 negara bagian', len(geoms) == 3)
for nd in (3, 2):
    fc = {'type': 'FeatureCollection', 'features': []}
    for g in geoms:
        polys = [g['arcs']] if g['type'] == 'Polygon' else g['arcs']
        fc['features'].append({'type': 'Feature', 'properties': {'name': wanted[g['id']]},
                               'geometry': {'type': 'MultiPolygon', 'coordinates': [[rounded(ring(r), nd) for r in poly] for poly in polys]}})
    text = json.dumps(fc, separators=(',', ':'))
    if len(text) <= 50_000:
        break
check('states: ≤ 50 KB', len(text) <= 50_000, f'{len(text):,} byte, {nd} desimal')
open(f'{DATA}/states.json', 'w').write(text)
PROV['states.json'] = {'source': ['npm us-atlas@3.0.1 states-10m.json (lisensi ISC), dari US Census Bureau cartographic boundary files 2017'], 'rows': 3}

PASS  states: 3 negara bagian
PASS  states: ≤ 50 KB  [16,022 byte, 3 desimal]


In [11]:
# Catatan peta dihitung dari data: titik berulang = sel ±10 km (0,1°) yang berisi ≥ 10 ledakan.
ex['c10'], ex['l10'] = (ex.lat * 10).round() / 10, (ex.lon * 10).round() / 10
ex['repeat'] = ex.groupby(['c10', 'l10']).event_id.transform('size') >= 10
lt = pd.to_datetime(ex.local_time); ex['office'] = (lt.dt.dayofweek < 5) & lt.dt.hour.between(8, 16)
idn = lambda v: f'{v:.0f}'
EX_NOTES = {}
for k in ['K4', 'K5']:
    g = ev_st.loc[ev_st.group == k, ['event_id']].merge(ex[['event_id', 'repeat', 'office']], on='event_id')
    EX_NOTES[k] = (f'{idn(100 * g.office.mean())}% ledakan terjadi pada jam kerja dan {idn(100 * g["repeat"].mean())}% di titik yang berulang, '
                   'pola yang mengarah ke peledakan tambang/<em>quarry</em> (belum dikonfirmasi).')
v = volc_check.set_index(['kelompok', 'gunung'])
n30 = lambda k, g: int(v.loc[(k, g), 'stasiun_≤30km'])
EX_NOTES['K2'] = f"{n30('K2', 'Mount St. Helens')} stasiun berada dalam 30 km dari Mount St. Helens dan {n30('K2', 'Mount Hood')} dari Mount Hood."
EX_NOTES['K6'] = f"{n30('K6', 'Newberry Volcano')} dari {int(grp.loc['K6', 'stations'])} stasiun berada dalam 30 km dari Newberry Volcano."
for k, t_ in EX_NOTES.items(): print(k, t_)

n_bc = int((ex.lat > 49).sum()); w = ex[ex.lat.between(42, 45.2) & ex.lon.between(-124.5, -121.8)]
OBS = '\n'.join(['## What the explosion layer shows', '',
    f'- Western Oregon box (42–45.2°N, 124.5–121.8°W): {len(w):,} explosions, {100 * w.office.mean():.1f}% in weekday working hours, {100 * w["repeat"].mean():.1f}% at repeated sites. They spread along the western valleys and repeat at many sites. There is no single dominant point.',
    '- K4 and K5 notes: see `summary.json → region_notes`. The wording says "possible quarry blasting, not confirmed": the pattern fits, but the data has no blast-source field.',
    f'- The densest cluster overall is outside the US: {n_bc:,} explosions ({100 * n_bc / len(ex):.1f}%) lie north of 49°N, mostly near 49.4°N, 120.5°W (British Columbia). K1 stations record them. The map extends north of the border to show this.',
    '- §7.5 says K6 is "near the Three Sisters". Not supported: the K6 centre is 101 km away and 0 of 22 stations are within 30 km. Only the Newberry Volcano note is used.', ''])
PROV['summary.json'] = {'source': ['/datalake/graph/g1_edges', f'{PROCESSED}/metadata', f'{DASH}/lpa_rerun'], 'rows': 1}
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=ROOT).decode().strip()
summary = {
    'provenance': {'generated_at': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'), 'git_commit': commit, 'files': PROV},
    'counts': {'test_recordings': len(p), 'g1_events': len(deg), 'g1_stations': int(st.shape[0]), 'g2_edges': e2.count(), 'g1_edges': e1.count()},
    'stations_per_event': per_event,
    'office_hours_share': office,
    'groups': groups,
    'network_average_pct': {c: float(net[c]) for c in EVENTS},
    'region_notes': EX_NOTES,
    'volcanoes': [{'name': n, 'lat': la, 'lon': lo} for n, la, lo in VOLCANOES],
    'volcano_source': 'List of Cascade volcanoes (Wikipedia), coordinates as listed',
}
json.dump(summary, open(f'{DATA}/summary.json', 'w'), ensure_ascii=False, indent=1)

sizes_b = {f: os.path.getsize(f'{DATA}/{f}') for f in sorted(os.listdir(DATA))}
total_b = sum(sizes_b.values())
check('docs/data: total < 10 MB', total_b < 10_000_000, f'{total_b / 1e6:.2f} MB')
check('summary: provenance mencakup semua file', set(PROV) == set(sizes_b), sorted(PROV))
fails = [c for c in CHECKS if c[1] == 'FAIL']
print(f'{len(CHECKS)} pemeriksaan, {len(fails)} FAIL (non-hard):', fails)
sizes_b

K4 91% ledakan terjadi pada jam kerja dan 47% di titik yang berulang, pola yang mengarah ke peledakan tambang/<em>quarry</em> (belum dikonfirmasi).
K5 90% ledakan terjadi pada jam kerja dan 49% di titik yang berulang, pola yang mengarah ke peledakan tambang/<em>quarry</em> (belum dikonfirmasi).
K2 35 stasiun berada dalam 30 km dari Mount St. Helens dan 12 dari Mount Hood.
K6 14 dari 22 stasiun berada dalam 30 km dari Newberry Volcano.
PASS  docs/data: total < 10 MB  [3.32 MB]
PASS  summary: provenance mencakup semua file  [['explosions.csv', 'predictions.csv', 'states.json', 'stations.csv', 'summary.json', 'waveforms_sample.json']]
59 pemeriksaan, 1 FAIL (non-hard): [('network_average surface_event vs PRD ±0,05', 'FAIL', '4.7 vs 4.8')]


{'explosions.csv': 310799,
 'predictions.csv': 2794751,
 'states.json': 16022,
 'stations.csv': 38648,
 'summary.json': 5613,
 'waveforms_sample.json': 156828}

## 6. Laporan ekspor

In [12]:
md = lambda df: df.to_markdown(index=False)
rep = ['# Phase 1 — Export report', '', f"Generated {summary['provenance']['generated_at']} by `notebooks/13_dashboard_export.ipynb` (git {commit}).", '',
       '## Files in `docs/data/`', '', md(pd.DataFrame([{'file': f, 'bytes': b, 'rows': PROV[f]['rows'], 'source': '; '.join(PROV[f]['source'])} for f, b in sizes_b.items()])),
       '', f'Total: {total_b / 1e6:.2f} MB (limit 10 MB).', '',
       '## New paths (nothing existing was changed)', '', f'- HDFS `{DASH}/lpa_rerun`', '- `docs/data/*`', '- `outputs/dashboard/*`', '- `notebooks/13_dashboard_export.ipynb`', '',
       '## Deviations from the report numbers', '',
       f"- Network average surface event = {net['surface_event']}% (PRD 4.8%). Exact value 8,912 / 187,751; 4.8 came from averaging rounded group percentages.",
       '- `pct_*` in `stations.csv` are percentages 0–100 with 1 decimal, matching `summary.json → groups[].pct`.',
       '- Label propagation: mapping identical to saved `/datalake/graph/g2_communities` (PRD thought it was never saved).', '',
       '## Waveform sample buckets (recordings taken)', '', md(cover.pivot(index='kelas', columns='keranjang', values='diambil').loc[CLASSES].reset_index()), '',
       '## Label propagation re-run (tolerance rule §6.3)', '', md(grp.loc[KEYS].round(2).reset_index()), '',
       '## Explosion locations and station groups', '', md(ex_by_group), '', 'Densest 0.25° cells:', '', md(cells.head(10).round(2).reset_index()), '', '## Volcano distance checks', '', md(volc_check), '',
       OBS, '', '## All checks', '', md(pd.DataFrame(CHECKS, columns=['check', 'result', 'detail'])), '']
open(f'{AUDIT}/01_export_report.md', 'w').write('\n'.join(rep))
sh(f'hdfs dfs -ls {DASH}');

$ hdfs dfs -ls /datalake/dashboard
Found 1 items
drwxrwx---+  - axelnakata engineer          0 2026-11-09 12:20 /datalake/dashboard/lpa_rerun


In [13]:
spark.stop()